# S03 · Aggregate at the correct grain

**Outcome:** Prevent duplicate joins and distinguish COUNT(*) from COUNT of a bound value.

**Time:** about 40 minutes. Run cells in order. Edit the exercise cell after completing the walkthrough.

SPARQL results are multisets. Joining a record to several related terms can produce several rows for one record. COUNT(?r) then counts appearances. COUNT(DISTINCT ?r) counts unique RDF terms, which is often the intended record count. It does not automatically merge two names that a reasoner considers equal unless the query implementation exposes that equivalence.

COUNT(?value) ignores unbound values. COUNT(*) counts the surviving solution row even when an OPTIONAL value is absent. Use this distinction to report both encounters and encounters with a recorded A1C category. For more complex products, preaggregate each independent one-to-many branch in a subquery and join the grouped results at their common grain.

GROUP_CONCAT ordering is not portable unless a specific implementation contract guarantees it. SAMPLE chooses an arbitrary member. Do not use either as a deterministic evidence selection rule. A result that happens to stay stable during a small local run is not a specification guarantee.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Count all rows and available values

In [2]:
g=build_asserted()
result=list(query(g,'SELECT (COUNT(*) AS ?rows) (COUNT(?a) AS ?available) WHERE { ?r a ex:EncounterRecord OPTIONAL { ?r ex:a1cCategory ?a } }'))
assert tuple(int(v) for v in result[0])==(60,18)
display(result)

[(rdflib.term.Literal('60', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),
  rdflib.term.Literal('18', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))]

## See a cross-product inflate the count

In [3]:
inflated=list(query(g,'SELECT (COUNT(?r) AS ?n) (COUNT(DISTINCT ?r) AS ?unique) WHERE { ?r a ex:EncounterRecord . VALUES ?view { "source" "review" } }'))
assert tuple(int(v) for v in inflated[0])==(120,60)
display(inflated)

[(rdflib.term.Literal('120', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),
  rdflib.term.Literal('60', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))]

## Your turn

Return the three code-group counts using GROUP BY and COUNT(DISTINCT ?r), as a dictionary keyed by the last segment of the code IRI.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = None  # Write your solution here

In [5]:
learner_check(answer, lambda x:x=={'250.02':20,'428':20,'493':20}, 'Group by the code IRI; do not join labels before counting.')

Exercise not completed. Group by the code IRI; do not join labels before counting.
Out[0]: False


## Explain your model

How would you show zero counts for allowed categories with no encounters?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** _Write your explanation here._